# Summarization

## 1. Setup and Library Installation

In [1]:
!pip install transformers datasets accelerate sentence-splitter sumy evaluate rouge-score bert-score
!pip install --upgrade accelerate # Necessary for newer versions of Trainer
!pip install nltk

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.3/97.3 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 53.8 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=84e0e6771184d6caead8a4e377100d7d28c8c92b668b4c67afbca3d7c939a525
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
  Created wheel for breadability: filename=breadability-0.1.20-py2.py3-none-any.whl size=21695 sha256=a2e958d6257ace3417973e4ff16637636e48284e302ffb7de2ba63f5782fb28b
  Stored in directory: /root/.cache/pip/wheels/32/99/6

In [2]:
import pandas as pd
import numpy as np
import nltk
from tqdm.auto import tqdm
import os
import random
import torch
from datasets import Dataset
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    PegasusTokenizer,
    PegasusForConditionalGeneration,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
import evaluate

# --- Extractive Summarization Libraries ---
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer as SumyTokenizer
from sumy.summarizers.text_rank import TextRankSummarizer
from sentence_splitter import SentenceSplitter

In [4]:
from google.colab import drive
import os

drive.mount('/content/drive/')
os.chdir("/content/drive/MyDrive/Tubes/Summarization/")

Mounted at /content/drive/


## 2. Configuration and Data Loading

In [5]:
# NLTK data for sentence tokenization
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Configuration
RANDOM_SEED = 42
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128

# Model Checkpoints
MODEL_CHECKPOINTS = {
    "BART": "facebook/bart-base",
    "PEGASUS": "google/pegasus-large", # Using large model for strong abstractive comparison
}
CHECKPOINT_PATHS = {
    "BART_FULL": "experiment_full/bart_finetuned_summarizer",
    "PEGASUS_FULL": "experiment_full/pegasus_finetuned_summarizer",
    "BART_SAMPLED": "experiment_sampled/bart_finetuned_summarizer",
    "PEGASUS_SAMPLED": "experiment_sampled/pegasus_finetuned_summarizer"
}

# Sampling Parameters for Training Data
LONG_REVIEW_SAMPLE_RATIO = 0.9
LONG_REVIEW_CUTOFF_PERCENTILE = 0.60

# Data File Paths
TRAIN_FILE = 'dataset_summaries.csv'
TEST_FILE = 'gold_standard_summaries.csv'

# Column Names
INPUT_COL = 'english_translation'
TRAIN_TARGET_COL = 'summary'
TEST_TARGET_COL = 'summary'
ID_COL = 'id'

In [6]:
def prepare_datasets(df_train_raw, df_test_raw):
    """
    1. Filters and cleans training data, excluding any IDs present in the test data.
    2. Calculates text length and performs stratified sampling on the training data.
    """
    # 1. Filter out data leakage (Exclusion)
    test_ids = set(df_test_raw[ID_COL].unique())
    df_clean = df_train_raw[~df_train_raw[ID_COL].isin(test_ids)].copy()

    # Filter rows with missing input/target columns
    df_clean.dropna(subset=[INPUT_COL, TRAIN_TARGET_COL], inplace=True)

    # 2. Stratified Sampling for Training Set

    # Calculate word count for length stratification
    df_clean['text_length'] = df_clean[INPUT_COL].apply(lambda x: len(str(x).split()))

    # Determine the length cutoff based on the desired percentile
    length_cutoff = df_clean['text_length'].quantile(LONG_REVIEW_CUTOFF_PERCENTILE)

    # Stratify the data into Long and Short groups
    df_long = df_clean[df_clean['text_length'] >= length_cutoff]
    df_short = df_clean[df_clean['text_length'] < length_cutoff]

    # Determine target sample sizes (We sample the entire filtered training set,
    # but use replace=True/False to manage size if necessary. Here, we sample all.)
    total_samples = len(df_clean)
    long_sample_size = int(total_samples * LONG_REVIEW_SAMPLE_RATIO)
    short_sample_size = total_samples - long_sample_size

    # Adjust sizes if strata are too small (using simple min check)
    long_sample_size = min(long_sample_size, len(df_long))
    short_sample_size = min(short_sample_size, len(df_short), round(long_sample_size * (1-LONG_REVIEW_SAMPLE_RATIO) / LONG_REVIEW_SAMPLE_RATIO))
    df_long_sample = df_long.sample(n=long_sample_size, random_state=RANDOM_SEED)
    df_short_sample = df_short.sample(n=short_sample_size, random_state=RANDOM_SEED)

    # Combine the samples (Training set now biased towards long reviews)
    df_final_train = pd.concat([df_long_sample, df_short_sample]).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

    print(f"Total Rows in Raw Training Data: {len(df_train_raw)}")
    print(f"Number of rows excluded (Data Leakage): {len(df_train_raw) - len(df_clean)}")
    print(f"Final Training Size (No Leakage, Stratified): {len(df_final_train)}")
    print(f"  - Long Reviews (over {length_cutoff:.0f} words): {long_sample_size}")
    print(f"  - Short Reviews: {short_sample_size}")

    # Convert to Hugging Face Dataset
    train_dataset = Dataset.from_pandas(df_final_train)
    return df_final_train, train_dataset

In [7]:
def preprocess_function(examples, tokenizer, input_col, target_col):
    """Tokenizes inputs and labels for Seq2Seq model."""
    model_inputs = tokenizer(
        examples[input_col],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding="max_length"
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples[target_col],
            max_length=MAX_TARGET_LENGTH,
            truncation=True,
            padding="max_length"
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

## 3. Extractive Baselines (Experiment A & B)

In [8]:
def run_extractive_baselines(df_test):
    """Generates summaries for the two extractive baselines."""
    def extract_first_sentence(text):
        if pd.isna(text) or not text.strip(): return ""
        sentences = nltk.sent_tokenize(text)
        return sentences[0].strip() if sentences else ""

    def summarize_textrank(text, sentence_count=2):
        if pd.isna(text) or not text.strip(): return extract_first_sentence(text)
        try:
            parser = PlaintextParser.from_string(text, SumyTokenizer("english"))
            summarizer = TextRankSummarizer()
            splitter = SentenceSplitter(language='en')
            sentences = splitter.split(text)
            num_sentences = min(len(sentences), sentence_count)
            summary = summarizer(parser.document, num_sentences)
            return " ".join([str(s) for s in summary])
        except Exception:
            return extract_first_sentence(text)

    print("\nStarting Extractive Baselines (Exp A & B)...")
    tqdm.pandas(desc="Running Exp A: First Sentence")
    df_test['summary_ext_first_sentence'] = df_test[INPUT_COL].progress_apply(extract_first_sentence)
    tqdm.pandas(desc="Running Exp B: TextRank")
    df_test['summary_ext_textrank'] = df_test[INPUT_COL].progress_apply(summarize_textrank, sentence_count=2)
    print("Extractive baselines completed.")
    return df_test

## 4. Fine-Tuning and Abstractive Generation (Experiment C & D)

### Fine-Tuning

In [9]:
def run_fine_tuning(model_name, model_path, train_dataset):
    """Fine-tunes the specified model and saves the checkpoint."""

    # --- A. Initialization ---
    if model_name == "BART":
        tokenizer = BartTokenizer.from_pretrained(model_path)
        model = BartForConditionalGeneration.from_pretrained(model_path).to('cuda' if torch.cuda.is_available() else 'cpu')
    elif model_name == "PEGASUS":
        tokenizer = PegasusTokenizer.from_pretrained(model_path)
        model = PegasusForConditionalGeneration.from_pretrained(model_path).to('cuda' if torch.cuda.is_available() else 'cpu')
    else:
        raise ValueError(f"Unknown model: {model_name}")

    # --- B. Data Preparation ---
    # The columns to remove here are determined by the prepare_datasets function,
    # ensuring only necessary columns remain in the dataset object itself.
    tokenized_train_dataset = train_dataset.map(
        lambda x: preprocess_function(x, tokenizer, INPUT_COL, TRAIN_TARGET_COL),
        batched=True,
        # Removing columns explicitly set in prepare_datasets
        remove_columns=[ID_COL, INPUT_COL, TRAIN_TARGET_COL]
    )

    print(f"Tokenization for {model_name} training complete.")

    # --- C. Training Setup and Execution ---
    output_dir = CHECKPOINT_PATHS[model_name]

    # Check if checkpoint already exists
    if os.path.exists(output_dir):
        print(f"Checkpoint for {model_name} already exists at {output_dir}. Skipping fine-tuning.")
        return

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        warmup_steps=500,
        weight_decay=0.01,
        logging_dir=f'./logs/{model_name.lower()}',
        logging_steps=200,
        save_strategy="epoch",
        fp16=torch.cuda.is_available(),
    )

    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )

    print(f"Starting Fine-Tuning for {model_name}...")
    trainer.train()
    trainer.save_model(output_dir)
    print(f"Fine-Tuning for {model_name} Complete. Checkpoint saved to {output_dir}")

### Generation

In [10]:
def run_generation(model_name, model_path, checkpoint_path, df_test, exp_type="ZERO_SHOT"):
    """
    Loads model (base model for ZERO_SHOT or checkpoint for FINETUNED) and generates summaries.

    Args:
        model_name (str): 'BART' or 'PEGASUS'.
        model_path (str): Path to the base model (e.g., 'facebook/bart-base').
        checkpoint_path (str): Path to the fine-tuned checkpoint.
        df_test (DataFrame): DataFrame to add the new summary column to.
        exp_type (str): "ZERO_SHOT" (Exp C) or "FINETUNED" (Exp D).
    """

    if model_name == "BART":
        tokenizer_class = BartTokenizer
        model_class = BartForConditionalGeneration
    elif model_name == "PEGASUS":
        tokenizer_class = PegasusTokenizer
        model_class = PegasusForConditionalGeneration
    else:
        raise ValueError(f"Unknown model: {model_name}")

    if exp_type == "FINETUNED_SAMPLED" and os.path.exists(checkpoint_path):
        load_path = checkpoint_path
        output_col = f'summary_abs_finetuned_{model_name}_SAMPLED'
        print(f"Loading fine-tuned model for {model_name} from: {load_path}")
    elif exp_type == "FINETUNED_FULL":
        load_path = checkpoint_path
        output_col = f'summary_abs_finetuned_{model_name}_FULL'
    else:
        load_path = model_path
        output_col = f'summary_abs_zero_shot_{model_name}'
        if exp_type == "FINETUNED":
             print(f"Checkpoint not found for {model_name}. Running Zero-Shot instead for Exp D column.")

    tokenizer = tokenizer_class.from_pretrained(model_path)
    model = model_class.from_pretrained(load_path).to('cuda' if torch.cuda.is_available() else 'cpu')

    input_texts = df_test[INPUT_COL].tolist()
    generated_summaries = []

    # Prepare inputs
    tokenized_inputs = tokenizer(
        input_texts,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=True,
        return_tensors="pt"
    )

    device = model.device
    tokenized_inputs = {k: v.to(device) for k, v in tokenized_inputs.items()}

    batch_size = 8
    model.eval()

    description = f"Generating {output_col}"

    for i in tqdm(range(0, len(input_texts), batch_size), desc=description):
        batch_inputs = {k: v[i:i + batch_size] for k, v in tokenized_inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **batch_inputs,
                max_length=MAX_TARGET_LENGTH,
                min_length=10,
                length_penalty=2.0,
                num_beams=4,
                early_stopping=True,
            )

        decoded_summaries = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        generated_summaries.extend(decoded_summaries)

    df_test[output_col] = generated_summaries
    print(f"Generation for {output_col} complete.")
    return df_test

### Evaluation

In [24]:
def evaluate_models(df_test):
    """Evaluates all generated summaries against the Gold Standard."""
    rouge_metric = evaluate.load("rouge")
    bertscore_metric = evaluate.load("bertscore")

    # List of all generated summary columns to evaluate
    GENERATED_SUMMARY_COLS = [
        'summary_ext_first_sentence',
        'summary_ext_textrank',
        'summary_abs_zero_shot_BART',
        'summary_abs_finetuned_BART_FULL',
        'summary_abs_finetuned_BART_SAMPLED',
        'summary_abs_zero_shot_PEGASUS',
        'summary_abs_finetuned_PEGASUS_FULL',
        'summary_abs_finetuned_PEGASUS_SAMPLED',
        'summary_decoder_gemini_flash_lite'
    ]

    # Filter for columns that actually exist after generation runs
    existing_cols = [col for col in GENERATED_SUMMARY_COLS if col in df_test.columns]

    # Convert references to a list of lists (required by some versions of the ROUGE metric)
    reference_summaries = [[r] for r in df_test[TEST_TARGET_COL].tolist()]
    evaluation_results = {}

    print("\n--- Running ROUGE Evaluation ---")
    for col in tqdm(existing_cols, desc="ROUGE"):
        candidate_summaries = df_test[col].tolist()

        results = rouge_metric.compute(
            predictions=candidate_summaries,
            references=reference_summaries,
            use_stemmer=True
        )

        # FIX: Directly access the fmeasure value from the returned dictionary.
        # This handles both the case where it returns Score object and the raw float
        # by defaulting to the direct float/dict access if 'mid' is not present.

        # The common return format is {'rouge1': {'low': ..., 'mid': {'fmeasure': 0.XX, ...}, 'high': ...}}
        # But when references is a list of lists, it should return a flat score.

        # We check the structure and handle it.

        if isinstance(results['rouge1'], dict) and 'mid' in results['rouge1']:
            # Older nested object format
            rouge_1 = results['rouge1']['mid']['fmeasure']
            rouge_2 = results['rouge2']['mid']['fmeasure']
            rouge_l = results['rougeL']['mid']['fmeasure']
        else:
            # New format (direct float value or dict with precision/recall/fmeasure keys)
            # Since the references list is converted to a list of single-item lists,
            # it should return the structure {'rouge-1': score, 'rouge-2': score...} where score is a float.
            rouge_1 = results['rouge1']
            rouge_2 = results['rouge2']
            rouge_l = results['rougeL']

        evaluation_results[col] = {
            'ROUGE-1': rouge_1,
            'ROUGE-2': rouge_2,
            'ROUGE-L': rouge_l
        }

    print("\n--- Running BERTScore Evaluation ---")
    for col in tqdm(existing_cols, desc="BERTScore"):
        candidate_summaries = df_test[col].tolist()

        results = bertscore_metric.compute(
            predictions=candidate_summaries,
            references=df_test[TEST_TARGET_COL].tolist(),
            model_type="distilbert-base-uncased",
            lang="en"
        )

        avg_f1 = np.mean(results['f1'])
        evaluation_results[col]['BERTScore-F1'] = avg_f1

    # --- Display Results ---
    df_results = pd.DataFrame.from_dict(evaluation_results, orient='index')
    df_results = (df_results * 100).round(2) # Convert to percentage and round
    df_results.index.name = "Model"

    print("\n\n------------------------------------------------------")
    print("FINAL AUTOMATED EVALUATION RESULTS (F1-Score, %)")
    print("------------------------------------------------------")
    print(df_results)
    print("------------------------------------------------------")

    # Save the full test results for analysis
    df_test.to_csv("summary_experiment_test_results_final.csv", index=False)
    print("Full results saved to summary_experiment_test_results_final.csv for human review.")

## EXECUTION

In [12]:
# Set seed for reproducibility
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

### 1. Data Loading and Preparation

In [13]:
try:
    df_train_raw = pd.read_csv(TRAIN_FILE)
    df_test = pd.read_csv(TEST_FILE)
except FileNotFoundError as e:
    print(f"ERROR: Could not find required data file: {e}")
    # Placeholder data to prevent crash if running locally without data
    if 'cuda' in torch.device('cuda').type: torch.cuda.empty_cache()
    raise SystemExit(1)

df_train_final, train_dataset = prepare_datasets(df_train_raw, df_test)

Total Rows in Raw Training Data: 6000
Number of rows excluded (Data Leakage): 200
Final Training Size (No Leakage, Stratified): 2603
  - Long Reviews (over 30 words): 2343
  - Short Reviews: 260


### 2. Extractive Baselines (Experiment A & B)


In [14]:
df_test = run_extractive_baselines(df_test)


Starting Extractive Baselines (Exp A & B)...


Running Exp A: First Sentence:   0%|          | 0/200 [00:00<?, ?it/s]

Running Exp B: TextRank:   0%|          | 0/200 [00:00<?, ?it/s]

Extractive baselines completed.


### 3. Fine-Tuning

In [ ]:
# run_fine_tuning("BART", MODEL_CHECKPOINTS["BART"], train_dataset)
# run_fine_tuning("PEGASUS", MODEL_CHECKPOINTS["PEGASUS"], train_dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: aureliusjustin (aureliusjustin-institut-teknologi-bandung) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
200,9.968200
400,6.649200
600,0.746200
800,0.451300
1000,0.398100
1200,0.410400
1400,0.378200
1600,0.364200
1800,0.350200


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 256, 'num_beams': 8, 'length_penalty': 0.8}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


Fine-Tuning for PEGASUS Complete. Checkpoint saved to ./pegasus_finetuned_summarizer


### 4. Generation




In [15]:
df_test = run_generation("BART", MODEL_CHECKPOINTS["BART"], None, df_test, exp_type="ZERO_SHOT")
df_test = run_generation("PEGASUS", MODEL_CHECKPOINTS["PEGASUS"], None, df_test, exp_type="ZERO_SHOT")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-large and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

Generating summary_abs_zero_shot_PEGASUS:   0%|          | 0/25 [00:00<?, ?it/s]

Generation for summary_abs_zero_shot_PEGASUS complete.


In [17]:
df_test = run_generation("BART", MODEL_CHECKPOINTS["BART"], CHECKPOINT_PATHS["BART_FULL"], df_test, exp_type="FINETUNED_FULL")
df_test = run_generation("PEGASUS", MODEL_CHECKPOINTS["PEGASUS"], CHECKPOINT_PATHS["PEGASUS_FULL"], df_test, exp_type="FINETUNED_FULL")

Generating summary_abs_finetuned_BART_FULL:   0%|          | 0/25 [00:00<?, ?it/s]

Generation for summary_abs_finetuned_BART_FULL complete.


Generating summary_abs_finetuned_PEGASUS_FULL:   0%|          | 0/25 [00:00<?, ?it/s]

Generation for summary_abs_finetuned_PEGASUS_FULL complete.


In [18]:
df_test = run_generation("BART", MODEL_CHECKPOINTS["BART"], CHECKPOINT_PATHS["BART_SAMPLED"], df_test, exp_type="FINETUNED_SAMPLED")
df_test = run_generation("PEGASUS", MODEL_CHECKPOINTS["PEGASUS"], CHECKPOINT_PATHS["PEGASUS_SAMPLED"], df_test, exp_type="FINETUNED_SAMPLED")

Loading fine-tuned model for BART from: experiment_sampled/bart_finetuned_summarizer


Generating summary_abs_finetuned_BART_SAMPLED:   0%|          | 0/25 [00:00<?, ?it/s]

Generation for summary_abs_finetuned_BART_SAMPLED complete.
Loading fine-tuned model for PEGASUS from: experiment_sampled/pegasus_finetuned_summarizer


Generating summary_abs_finetuned_PEGASUS_SAMPLED:   0%|          | 0/25 [00:00<?, ?it/s]

Generation for summary_abs_finetuned_PEGASUS_SAMPLED complete.


In [21]:
df_gemini_flash_lite = pd.read_csv('dataset_summaries.csv')

In [22]:
df_gemini_flash_lite_summaries = df_gemini_flash_lite[[ID_COL, TRAIN_TARGET_COL]].rename(columns={TRAIN_TARGET_COL: 'summary_decoder_gemini_flash_lite'}) # Assuming 'summary' is the target column in df_gemini_flash_lite
df_test = df_test.merge(df_gemini_flash_lite_summaries, on=ID_COL, how='left')

,id,datetime,location,text,rating,accessibility,facility,activity,english_translation,text_length,summary,summary_ext_first_sentence,summary_ext_textrank,summary_abs_zero_shot_BART,summary_abs_zero_shot_PEGASUS,summary_abs_finetuned_BART_FULL,summary_abs_finetuned_PEGASUS_FULL,summary_abs_finetuned_BART_SAMPLED,summary_abs_finetuned_PEGASUS_SAMPLED,summary_decoder_gemini_flash_lite
0,000cd22fbba5431fa57c09c4ca464c74,2022-02-15 13:43:46,Sanghyang Heuleut,"Berada jauh dr jalan raya , perlu jalan kaki n...",5,neutral,negative,positive,"Located far from the highway, it requires walk...",47,Access to this remote site requires a challeng...,"Located far from the highway, it requires walk...","Located far from the highway, it requires walk...","Located far from the highway, it requires walk...",Don't forget to bring water and food supplies ...,The attraction is easily accessible from the h...,The attraction requires a one-hour walk to rea...,"This attraction, located far from the highway,...",This attraction requires a one-hour walk from ...,Accessing this site involves a challenging one...
1,005582ccdb6e4346a0662775457da0cb,2023-02-13 13:24:34,Gunung Putri Lembang,"Tempat parkir luas, motor, helm dan mobil dipa...",4,neutral,positive,positive,"Spacious parking lot, motorcycles, helmets, an...",48,The attraction provides ample and secure parki...,"Spacious parking lot, motorcycles, helmets, an...",It's better to come around 5 or 6 PM because i...,"Spacious parking lot, motorcycles, helmets, an...","The city light view at night is cool, and the ...",The attraction offers ample parking and is wel...,This attraction offers a spacious parking lot ...,This attraction offers ample parking and ample...,"This attraction offers spacious parking, motor...",The attraction offers spacious and safe parkin...
2,018ae7fd6fa847e89e227fe814b09b14,2020-02-23 18:34:59,Orchid Forest Cikole,"sukaa banget disini.. tempatnya tenang bgt, wa...",4,neutral,positive,positive,Love it here so much. The place is very peacef...,64,Access to the attraction is challenging due to...,Love it here so much.,Even though initially the road leading here wa...,Love it here so much. The place is very peacef...,"It's cool, the view is good, and on the way ba...","Despite a damaged road requiring repair, the s...",This peaceful attraction offers a pleasant exp...,The attraction offers a peaceful experience wi...,This peaceful location offers good views and a...,"Despite a damaged 1km stretch of road, the des..."
3,01f41d1df50d44f58f1d3cc14b5de09f,2022-02-15 12:24:49,Curug Tilu Leuwi Opat,Nuansanya sejuk. Ke tempat curug nya juga gak ...,4,positive,positive,positive,The nuances are cool. It's not far to the wate...,47,This affordable and easily accessible attracti...,The nuances are cool.,"It's not far to the waterfall, and the water i...",The nuances are cool. It's not far to the wate...,"Overall, it's perfect for relieving fatigue",This spacious attraction offers clean water an...,This attraction offers scenic views and clean ...,"This spacious, clean attraction offers camping...",This spacious and clean attraction is ideal fo...,This attraction is easily accessible and featu...
4,02852883d5224e4bad7f8003af722190,2023-02-02 17:57:45,Floating Market Lembang,Tempatnya bagus.... Pemandangan indah.. Banyak...,4,neutral,neutral,positive,The place is nice.... Beautiful scenery.. Lots...,52,While the scenic attraction offers numerous ph...,The place is nice.... Beautiful scenery.. Lots...,The place is nice.... Beautiful scenery.. Lots...,The place is nice.... Beautiful scenery.. Lots...,Lots of photo spots... It's just a shame the t...,The attraction offers beautiful scenery and nu...,The attraction offers beautiful scenery and nu...,The attraction offers beautiful scenery and nu...,The attraction offers beautiful scenery and nu...,Despite beautiful scenery and numerous photo o...


### 5. Automated Evaluation

In [25]:
evaluate_models(df_test)


--- Running ROUGE Evaluation ---


ROUGE:   0%|          | 0/9 [00:00<?, ?it/s]


--- Running BERTScore Evaluation ---


BERTScore:   0%|          | 0/9 [00:00<?, ?it/s]



------------------------------------------------------
FINAL AUTOMATED EVALUATION RESULTS (F1-Score, %)
------------------------------------------------------
                                       ROUGE-1  ROUGE-2  ROUGE-L  BERTScore-F1
Model                                                                         
summary_ext_first_sentence               21.56     5.19    15.54         76.40
summary_ext_textrank                     32.46     8.28    22.64         79.60
summary_abs_zero_shot_BART               35.07     9.20    23.78         80.33
summary_abs_finetuned_BART_FULL          48.47    18.65    35.95         86.88
summary_abs_finetuned_BART_SAMPLED       48.02    18.00    35.78         86.80
summary_abs_zero_shot_PEGASUS            24.44     5.75    18.02         77.08
summary_abs_finetuned_PEGASUS_FULL       43.53    15.83    32.99         85.66
summary_abs_finetuned_PEGASUS_SAMPLED    48.08    17.91    35.56         86.68
summary_decoder_gemini_flash_lite        51.68   